## Descarga de datos de la API de Google Maps

In [34]:
import googlemaps
import os
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv

# 1. Configuración Inicial
load_dotenv()
api_key = os.getenv('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=api_key)

In [35]:
import googlemaps
import os
import time
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv

# 1. Configuración Inicial
load_dotenv()
api_key = os.getenv('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=api_key)

# --- CONFIGURACIÓN DE PUNTOS CLAVE ---
origen_query = "Terminal de Ómnibus Mariano Moreno, Rosario"
# Radio central para la búsqueda (Echesortu)
radio_query = "MO Pet Shopping - Sucursal Echesortu"

# Geocodificamos el origen para obtener su Place ID y Coordenadas
res_origen_geo = gmaps.geocode(origen_query)
origen_id = res_origen_geo[0]['place_id']
origen_coords = res_origen_geo[0]['geometry']['location']

# Geocodificamos el centro del radio de búsqueda
res_radio = gmaps.geocode(radio_query)
location_busqueda = res_radio[0]['geometry']['location']

# 2. Definición de Categorías
categorias_busqueda = [
    {'label': 'Restaurante', 'type': 'restaurant', 'keyword': 'restaurant'}, 
    {'label': 'Museo', 'type': 'museum', 'keyword': 'museo'},
    {'label': 'Heladería', 'type': None, 'keyword': 'heladeria artesanal'},
    {'label': 'Cervecería', 'type': 'bar', 'keyword': 'cerveceria artesanal comida'},
    {'label': 'Hotel', 'type': 'hotel', 'keyword': 'hotel'},
]

datos_lugares = []
place_ids_vistos = set()

# --- FUNCIÓN AUXILIAR PARA PROCESAR DETALLES ---
def obtener_info_lugar(p_id, etiqueta):
    try:
        detalles = gmaps.place(
            place_id=p_id, 
            fields=['name', 'geometry', 'formatted_address', 'rating', 'user_ratings_total', 'opening_hours'],
            language='es'
        ).get('result', {})

        dias_semana = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
        horarios_semanales = {dia: [] for dia in dias_semana}
        periodos = detalles.get('opening_hours', {}).get('periods', [])

        if periodos:
            for p in periodos:
                idx_google = p['open']['day']
                idx_nuestro = (idx_google - 1) % 7 
                nombre_dia = dias_semana[idx_nuestro]
                h_abre = p['open']['time']
                h_cierra = p.get('close', {}).get('time', '24h')
                horarios_semanales[nombre_dia].append({
                    'apertura': f"{h_abre[:2]}:{h_abre[2:]}",
                    'cierre': f"{h_cierra[:2]}:{h_cierra[2:]}" if h_cierra != '24h' else '24h'
                })
            for dia in dias_semana:
                if not horarios_semanales[dia]: horarios_semanales[dia] = "Cerrado"
        else:
            horarios_semanales = "No disponible"

        return {
            'nombre': detalles.get('name'),
            'tipo': etiqueta,
            'coords': detalles.get('geometry', {}).get('location'),
            'direccion': detalles.get('formatted_address'),
            'puntaje': detalles.get('rating', 0),
            'resenas': detalles.get('user_ratings_total', 0),
            'horarios_semana': horarios_semanales
        }
    except:
        return None

# --- PASO 3: PROCESAR EL ORIGEN PRIMERO ---
print("Procesando punto de origen...")
info_origen = obtener_info_lugar(origen_id, "Punto de Partida")
if info_origen:
    datos_lugares.append(info_origen)
    place_ids_vistos.add(origen_id)

# --- PASO 4: BUCLE DE BÚSQUEDA CON PAGINACIÓN ---
print(f"Buscando lugares de interés en Rosario (hasta 60 por categoría)...")
for cat in categorias_busqueda:
    print(f"Buscando {cat['label']}...")
    
    # Primera llamada (Página 1)
    busqueda = gmaps.places_nearby(
        location=location_busqueda,
        radius=3000,
        type=cat['type'],
        keyword=cat['keyword'],
        language='es'
    )

    while True:
        # Procesar resultados de la página actual
        for item in busqueda.get('results', []):
            pid = item['place_id']
            if pid in place_ids_vistos: continue
            
            # Filtro de restaurante (evitar comida al paso)
            tipos = item.get('types', [])
            if cat['label'] == 'Restaurante' and ('meal_takeaway' in tipos and 'restaurant' not in tipos):
                continue

            lugar_procesado = obtener_info_lugar(pid, cat['label'])
            if lugar_procesado:
                datos_lugares.append(lugar_procesado)
                place_ids_vistos.add(pid)

        # Verificar si hay más páginas
        token = busqueda.get('next_page_token')
        if not token:
            break
            
        # Pausa obligatoria de 2 segundos para que Google valide el token
        time.sleep(2)
        busqueda = gmaps.places_nearby(page_token=token)

# 5. Visualización
df = pd.DataFrame(datos_lugares)
print(f"\nSe encontraron {len(df) - 1} lugares + el punto de origen.")
print(df[['nombre', 'tipo', 'puntaje']].head(15))

Procesando punto de origen...
Buscando lugares de interés en Rosario (hasta 60 por categoría)...
Buscando Restaurante...
Buscando Museo...
Buscando Heladería...
Buscando Cervecería...
Buscando Hotel...

Se encontraron 249 lugares + el punto de origen.
                                    nombre              tipo  puntaje
0                                  Rosario  Punto de Partida      3.8
1                             Los Jardines       Restaurante      4.3
2                                   Riomío       Restaurante      4.2
3                                 El Cairo       Restaurante      4.2
4                     Orquídea Restaurante       Restaurante      4.5
5                             Don Giuseppe       Restaurante      4.0
6                                 Gran Vía       Restaurante      4.0
7                    Gran Lago - Resto Bar       Restaurante      4.1
8                                 Casimiro       Restaurante      4.2
9                  Rock&Feller's Bv. Oroño      

In [36]:
# Reemplazamos el nombre Rosario por "Terminal de omnibus Mariano Moreno"
df.loc[df['nombre'] == 'Rosario', 'nombre'] = "Terminal de Ómnibus Mariano Moreno"

In [37]:
# Se guardan el dataframe completo en un archivo .txt para análisis posterior
df.to_csv('data/df_completo.txt', index=False, sep='\t')


In [38]:
import pandas as pd
import ast

# Carga del dataframe para análisis posterior
df = pd.read_csv('data/df_completo.txt', sep='\t')

# Convertimos los strings de nuevo a diccionarios
# Usamos literal_eval porque es más seguro que eval()
def safe_eval(val):
    try:
        if pd.isna(val) or val == "No disponible":
            return val
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return val

df['coords'] = df['coords'].apply(safe_eval)
df['horarios_semana'] = df['horarios_semana'].apply(safe_eval)

# Ahora ya puedes correr el código de transformación que te pasé antes
print(type(df.iloc[0]['coords'])) # Debería decir <class 'dict'>

<class 'dict'>


## Filtrado de los puntos de interés

In [39]:
# 1. Separamos el Punto de Origen y los hoteles
origen = df[df['tipo'] == 'Punto de Partida']

hoteles = df[df['tipo'] == 'Hotel'].copy()

# 2. Filtramos los lugares que NO son el origen ni hoteles
lugares_interes = df[(df['tipo'] != 'Punto de Partida') & (df['tipo'] != 'Hotel')].copy()

# 3. FILTRO DE CONFIANZA: 
# Solo consideramos lugares con más de 200 reseñas (podes ajustar este número)
lugares_interes = lugares_interes[lugares_interes['resenas'] >= 200]

# 4. CREAR IDENTIFICADOR DE MARCA
# Tomamos las primeras 4 palabras del nombre para detectar sucursales
lugares_interes['marca_id'] = lugares_interes['nombre'].str.split().str[:4].str.join(' ')
# Borramos las sucursales repetidas
lugares_interes = lugares_interes.drop_duplicates(subset=['tipo', 'marca_id'], keep='first')

# 5. FILTRO DE APERTURA MIÉRCOLES
def esta_abierto_miercoles(horarios):
    # Si no es un diccionario o es "No disponible", lo descartamos o tratamos como cerrado
    if not isinstance(horarios, dict):
        return False
    
    status_miercoles = horarios.get('Miércoles', 'Cerrado')
    
    # Si es una lista con datos, significa que tiene turnos de apertura
    if isinstance(status_miercoles, list) and len(status_miercoles) > 0:
        return True
    
    return False

# Aplicamos el filtro
lugares_abiertos = lugares_interes[lugares_interes['horarios_semana'].apply(esta_abierto_miercoles)]
# -----------------------------------------------

# 6. Agrupamos por 'tipo' y tomamos los 5 mejores de los que SÍ abren el miércoles (10 para los hoteles)
top_por_categoria = lugares_abiertos.sort_values(['tipo', 'puntaje'], ascending=[True, False])
top_5_lugares = top_por_categoria.groupby('tipo').head(5)

In [40]:
# Búsqueda de mejores 10 hoteles
lugares_interes = hoteles.copy()

# 1. FILTRO DE CONFIANZA: 
# Solo consideramos lugares con más de 200 reseñas (podes ajustar este número)
lugares_interes = lugares_interes[lugares_interes['resenas'] >= 200]

# 2. CREAR IDENTIFICADOR DE MARCA
# Tomamos las primeras 4 palabras del nombre para detectar sucursales
lugares_interes['marca_id'] = lugares_interes['nombre'].str.split().str[:4].str.join(' ')
# Borramos las sucursales repetidas
lugares_interes = lugares_interes.drop_duplicates(subset=['tipo', 'marca_id'], keep='first')

# 3. Agrupamos por 'tipo' y tomamos los 10 mejores
top_10_lugares = lugares_interes.sort_values('puntaje', ascending=False).head(10)

# 4. Definimos todos los posibles puntos de origen
depots = pd.concat([origen, top_10_lugares]).reset_index(drop=True)

In [41]:
# Guardamos el dataframe de depots en un archivo .txt para análisis posterior
depots.to_csv('data/depots.txt', index=False)

## Creación de g_nodos y matrices de distancia y tiempo para cada depot

In [ ]:
import numpy as np
import pandas as pd
import time
import os

# Hacemos un bucle para preparar un dataset según cada depot (origen + top 5 de cada categoría)
for i, depot in enumerate(depots.itertuples(index=False)):
    print(f"\nPreparando dataset para Depot {i+1}/{len(depots)}: {depot.nombre} ({depot.tipo})")
    df_final = pd.concat([pd.DataFrame([depot._asdict()]), top_5_lugares]).reset_index(drop=True)

    # 1. Extraer latitud y longitud de la columna 'coords'
    df_final['lat'] = df_final['coords'].apply(lambda x: x['lat'] if isinstance(x, dict) else np.nan)
    df_final['lon'] = df_final['coords'].apply(lambda x: x['lng'] if isinstance(x, dict) else np.nan)

    # 2. Función para extraer apertura y cierre del Miércoles
    def extraer_horario_miercoles(horarios, tipo_retorno):
        # Si es "No disponible" o no es un diccionario, devolvemos NA
        if not isinstance(horarios, dict) or horarios == "No disponible":
            return "NA"
    
        # Buscamos la entrada de 'Miércoles'
        miercoles = horarios.get('Miércoles', "Cerrado")
    
        # Si el lugar está cerrado ese día
        if miercoles == "Cerrado":
            return "Cerrado"
    
        # Si tiene datos (recordá que es una lista de turnos) tomamos el primero
        if isinstance(miercoles, list) and len(miercoles) > 0:
            if tipo_retorno == 'apertura':
                return miercoles[0]['apertura']
            else:
                return miercoles[0]['cierre']
            
        return "NA"

    # 3. Aplicar la función para crear las nuevas columnas
    df_final['Apertura'] = df_final['horarios_semana'].apply(lambda x: extraer_horario_miercoles(x, 'apertura'))
    df_final['Cierre'] = df_final['horarios_semana'].apply(lambda x: extraer_horario_miercoles(x, 'cierre'))

    # Si Apertura y Cierre son NA o cerrados, reemplazar apertura por 00:00 y cierre por 23:59
    df_final['Apertura'] = df_final['Apertura'].replace('NA', '00:00')
    df_final['Apertura'] = df_final['Apertura'].replace('Cerrado', '00:00')
    df_final['Cierre'] = df_final['Cierre'].replace('NA', '23:59')
    df_final['Cierre'] = df_final['Cierre'].replace('Cerrado', '23:59')

    # 4. Crear el ID (usando el índice) y seleccionar las columnas finales
    df_final['ID'] = df_final.index
    columnas_interes = ['ID', 'nombre', 'tipo', 'puntaje', 'lat', 'lon', 'Apertura', 'Cierre']
    df_tabla_final = df_final[columnas_interes].copy()

    # 5. Visualización
    print(df_tabla_final.head())

    # 6. Cambiamos el valor 'Punto de partida' de la columna tipo a 'depot'
    df_tabla_final['tipo'] = df_tabla_final['tipo'].replace({'Punto de Partida': 'depot'})
    df_tabla_final['tipo'] = df_tabla_final['tipo'].replace({'Hotel': 'depot'})

    # 7. Guardamos la tabla final en un nuevo archivo .txt para análisis posterior en cada carpeta por depot
       
    # Define the directory path
    directory = f'data/depot{i}'

    # Create the directory if it doesn't exist
    if not os.path.exists(directory):
        os.makedirs(directory)

    # Now save the file
    df_tabla_final.to_csv(f'{directory}/g_nodos.txt', index=False, sep='\t')

    ### MATRIZ DE DISTANCIAS Y TIEMPOS CON GOOGLE MAPS API
    # 1. Actualizamos datos lugares con los 21 lugares de interés (1 origen + 20 seleccionados)
    datos_lugares = df_final.to_dict('records')

    # 2. Preparar datos de los nodos
    coords_solo = [d['coords'] for d in datos_lugares]
    indices_solo = [d['ID'] for d in datos_lugares]
    n = len(coords_solo)

    # Matrices para almacenar valores puros
    matriz_segundos = []
    matriz_metros = []

    print(f"Calculando matriz de {n}x{n} (Valores puros: Segundos y Metros)...")

    # 3. Procesamiento por filas (Batching para evitar el límite de 100 elementos)
    for j in range(n):
        origen_actual = [coords_solo[j]]
    
        try:
            # Eliminamos departure_time para que NO considere tráfico
            res_fila = gmaps.distance_matrix(
                origins=origen_actual,
                destinations=coords_solo,
                mode="driving",
                language="es"
            )
        
            segundos_fila = []
            metros_fila = []
            elementos = res_fila['rows'][0]['elements']
        
            for el in elementos:
                if el['status'] == 'OK':
                    # Extraemos valores puros sin procesar
                    val_segundos = el['duration']['value'] # Segundos
                    val_metros = el['distance']['value']   # Metros
                
                    segundos_fila.append(val_segundos)
                    metros_fila.append(val_metros)
                else:
                    segundos_fila.append(None)
                    metros_fila.append(None)
        
            matriz_segundos.append(segundos_fila)
            matriz_metros.append(metros_fila)
        
            print(f"✔ Fila {j+1}/{n} completada.")
        
        except Exception as e:
            print(f"✘ Error en fila {j+1}: {e}")
            matriz_segundos.append([None] * n)
            matriz_metros.append([None] * n)

    # 4. Crear DataFrames con los valores crudos
    df_tiempo = pd.DataFrame(matriz_segundos, index=indices_solo, columns=indices_solo)
    df_distancia = pd.DataFrame(matriz_metros, index=indices_solo, columns=indices_solo)

    # 5. Guardamos resultados
    # Define the directory path
    directory = f'data/depot{i}'

    # Guardar df_distancias en un archivo CSV
    df_distancia.to_csv(f'{directory}/g_distancias.csv')

    # Guardar df_tiempos en un archivo CSV
    df_tiempo.to_csv(f'{directory}/g_tiempos.csv')




Preparando dataset para Depot 1/11: Terminal de Ómnibus Mariano Moreno (Punto de Partida)
   ID                                      nombre              tipo  puntaje  \
0   0          Terminal de Ómnibus Mariano Moreno  Punto de Partida      3.8   
1   1                    La Vizcachera cervecería        Cervecería      4.7   
2   2  Cerveza Patagonia - Refugio Casa Del Tango        Cervecería      4.7   
3   3                              Manush Rosario        Cervecería      4.6   
4   4                        Mosto Somos Cervezas        Cervecería      4.6   

         lat        lon Apertura Cierre  
0 -32.939376 -60.671065    00:00  23:59  
1 -32.950728 -60.676545    18:00  02:00  
2 -32.934723 -60.644414    17:00  01:00  
3 -32.932807 -60.652391    17:30  01:30  
4 -32.946557 -60.658378    18:00  01:00  
Calculando matriz de 20x20 (Valores puros: Segundos y Metros)...
✔ Fila 1/20 completada.
✔ Fila 2/20 completada.
✔ Fila 3/20 completada.
✔ Fila 4/20 completada.
✔ Fila 5/20 com

In [28]:
df_tiempo

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,918,733,508,383,288,308,799,792,1076,289,200,462,626,375,415,446,917,351,373
1,865,0,869,421,688,749,965,773,981,1002,645,773,796,955,807,980,466,332,783,513
2,741,587,0,753,450,595,479,137,212,388,686,598,310,469,1056,495,678,586,1035,753
3,616,479,699,0,396,464,762,764,757,1035,365,456,593,751,572,765,148,478,548,228
4,358,724,492,393,0,206,490,557,550,834,312,224,321,479,686,517,301,723,646,379
5,275,879,457,441,220,0,300,564,578,846,221,133,187,351,588,316,360,878,570,288
6,382,1200,613,803,559,491,0,749,624,1001,583,495,342,448,597,227,722,1198,562,650
7,794,479,187,704,482,648,652,0,355,382,716,652,483,642,1059,668,628,478,1036,681
8,718,835,292,707,405,572,451,447,0,680,663,575,282,287,1010,466,636,834,987,730
9,964,660,416,867,652,818,851,356,584,0,910,822,681,840,1240,866,791,659,1216,844


In [29]:
df_distancia

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,5233,4159,2582,1797,1405,1392,4385,4115,5854,1436,954,2259,3242,1563,2020,2168,5228,1532,1791
1,4713,0,4493,2478,3827,4221,5723,4401,5141,5670,3727,4355,4641,5681,4671,5818,2614,1730,4640,3022
2,4106,3637,0,4308,2410,3355,2994,737,1076,2559,3976,3493,1912,2952,5637,3089,3900,3632,5303,4334
3,2992,2687,4214,0,1857,2245,3802,4446,4176,5903,1749,2308,2719,3759,2908,3889,646,2682,2876,1046
4,1655,4061,2946,1900,0,909,2465,3177,2907,4647,1525,1042,1383,2422,3225,2552,1489,4056,2869,1883
5,1328,5192,2756,2344,991,0,1550,2975,2756,5315,1198,716,855,1839,2901,1645,1934,5187,2531,1556
6,1595,7202,3565,4180,2816,2415,0,4302,3177,6124,3035,2553,1665,2371,2626,1149,3771,7197,2508,3393
7,4313,2900,1035,4407,2652,3563,4032,0,1865,2328,4178,3701,2950,3990,5877,4127,4000,2895,5846,4394
8,3906,4623,1701,4148,2244,3155,2826,2585,0,4260,3776,3293,1744,1933,5477,2921,3738,4618,5446,4134
9,5383,3970,2530,5575,3722,4633,5512,1971,3710,0,5253,4771,4430,5469,6947,5607,5168,3964,6916,5562
